# MedPilot - vLLM + Qwen Server

Chay Qwen tren Colab voi vLLM + Ngrok tunnel

**CHAY TUAN TỰ: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8**

In [ ]:
# 1. Cai dat vLLM
!pip install vllm -q

In [ ]:
# 2. Tai va cai Ngrok
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip 2>/dev/null
!unzip -o ngrok.zip 2>/dev/null
!chmod +x ngrok

In [ ]:
# 3. Setup Ngrok Token
NGROK_TOKEN = "3BJuRXeiwgSNG5zWfm89ESHTDtg_3QBZ3CU3f7doQ3a6stpQS"
!./ngrok authtoken {NGROK_TOKEN}

In [ ]:
# 4. Import library
import subprocess
import time
import requests

In [ ]:
# 5. Start vLLM Server (nen - MAT 5-10 PHUT)
print("=" * 60)
print("STARTING vLLM...")
print("Model: Qwen/Qwen2.5-1.5B-Instruct")
print("DU CHO 5-10 PHUT DE LOAD MODEL!")
print("=" * 60)

# Kill port 8000 if exists
!pkill -f vllm 2>/dev/null
!lsof -ti:8000 | xargs kill -9 2>/dev/null
time.sleep(2)

# Start vLLM
vllm_process = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--host', '0.0.0.0',
    '--port', '8000',
    '--tensor-parallel-size', '1'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print("vLLM dang load model...")
print("CHO 60 GIAY...")
time.sleep(60)

print("Dang kiem tra...")
try:
    r = requests.get("http://localhost:8000/v1/models", timeout=5)
    print(f"vLLM Ready! Status: {r.status_code}")
except:
    print("vLLM chua san sang, cho them...")
    time.sleep(60)
    print("Thu lai...")

In [ ]:
# 6. Start Ngrok Tunnel
print("=" * 60)
print("STARTING NGROK...")
print("=" * 60)

# Kill existing ngrok
!pkill -f ngrok 2>/dev/null
time.sleep(2)

# Start ngrok (chay nen)
ngrok_process = subprocess.Popen([
    './ngrok', 'http', '8000',
    '--log', 'stdout'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print("Ngrok da start!")
print("CHO 10 GIAY...")
time.sleep(10)
print("=" * 60)

In [ ]:
# 7. Lay Public URL (RETRY NEU LOI)
print("=" * 60)
print("LAY PUBLIC URL")
print("=" * 60)

public_url = None

for i in range(5):
    print(f"Thu {i+1}/5...")
    try:
        response = requests.get("http://localhost:4040/api/tunnels", timeout=10)
        tunnels = response.json()["tunnels"]
        
        for tunnel in tunnels:
            if tunnel["proto"] == "https":
                public_url = tunnel["public_url"]
                print(f"\n>>> PUBLIC URL: {public_url}")
                print("\n" + "=" * 60)
                break
        
        if public_url:
            break
        
    except Exception as e:
        print(f"Loi: {str(e)[:50]}")
    
    time.sleep(5)

if not public_url:
    print("\n[ERROR] Khong lay duoc URL!")
    print("Kiem tra lai:")
    print("1. Ngrok co chay khong?")
    print("2. vLLM co load xong khong?")
else:
    print("\n>>> COPY URL SAU VAO .env:")
    print(f"\nVLLM_API_URL={public_url}/v1/chat/completions\n")

In [ ]:
# 8. Test API
if public_url:
    print("Testing API...")
    test_url = f"{public_url}/v1/chat/completions"
    
    payload = {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [{"role": "user", "content": "Xin chao?"}],
        "max_tokens": 50
    }
    
    try:
        response = requests.post(test_url, json=payload, timeout=120)
        if response.status_code == 200:
            data = response.json()
            answer = data["choices"][0]["message"]["content"]
            print(f"\n>>> API WORKING!")
            print(f"Response: {answer[:100]}")
        else:
            print(f"Error: {response.status_code}")
    except Exception as e:
        print(f"Loi: {e}")
else:
    print("Skip test - chua co URL")

---

## HUONG DAN:

1. **Chay tat ca o** theo thu tu 1 -> 8
2. **O 5:** Cho 5-10 phut de load model Qwen
3. **O 7:** Copy PUBLIC URL
4. **Paste vao .env** local:
   ```
   VLLM_API_URL=https://xxx.ngrok.io/v1/chat/completions
   ```

5. **Chay backend local:**
   ```
   python -m uvicorn app.main:app --port 8000
   ```